In [3]:
import requests, time, hashlib, csv
from pathlib import Path

RG_ID = "d9103c72-3807-4378-9ce7-b6f3e8fdd547"   # OK Computer
ALBUM = "to_pimp_a_butterfly"
HEADERS = {"User-Agent": "AlbumCoverProject/0.1 (akinbobola71@email.com)"}

out = Path("data/raw/candidate_fronts") / ALBUM
out.mkdir(parents=True, exist_ok=True)

# 1. List every release (edition) in the album
releases, offset = [], 0
while True:
    r = requests.get("https://musicbrainz.org/ws/2/release",
                     params={"release-group": RG_ID, "limit": 100, "offset": offset, "fmt": "json"},
                     headers=HEADERS, timeout=30)
    r.raise_for_status()
    data = r.json()
    releases += data["releases"]
    offset += 100
    if offset >= data["release-count"]:
        break
    time.sleep(1.1)
print(len(releases), "releases found")

# 2. Download the front image for each release, skipping exact duplicates
seen, rows = {}, []
for rel in releases:
    rid = rel["id"]
    row = {"release_id": rid, "date": rel.get("date", ""), "country": rel.get("country", ""),
           "has_front": False, "file": "", "duplicate_of": ""}
    caa = rel.get("cover-art-archive")
    if caa is None or caa.get("front"):
        time.sleep(1.1)
        r = requests.get(f"https://coverartarchive.org/release/{rid}", headers=HEADERS, timeout=30)
        if r.status_code == 200:
            front = next((i for i in r.json()["images"] if i.get("front")), None)
            if front:
                url = front["thumbnails"].get("500") or front["image"]
                img = requests.get(url, headers=HEADERS, timeout=60).content
                h = hashlib.sha256(img).hexdigest()
                row["has_front"] = True
                if h in seen:
                    row["duplicate_of"] = seen[h]
                else:
                    (out / f"{rid}.jpg").write_bytes(img)
                    seen[h] = f"{rid}.jpg"
                    row["file"] = f"{rid}.jpg"
    rows.append(row)

# 3. Save a manifest and print the feasibility numbers
with open(out / "manifest.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader()
    w.writerows(rows)

print("Releases total:          ", len(rows))
print("Releases with front art: ", sum(r["has_front"] for r in rows))
print("Distinct images (exact): ", len(seen))

23 releases found
Releases total:           23
Releases with front art:  21
Distinct images (exact):  20


In [1]:
import pandas as pd

df = pd.read_csv("data/metadata/screen.csv")

# 1. Overall status
print(df["status"].value_counts(), "\n")

# 2. Usable rows: matched, ok, and title looks right
df["title_ok"] = df["title_ok"].astype(str).str.lower() == "true"
ok = df[(df["status"] == "ok") & df["title_ok"]].copy()
ok["front_art"] = ok["front_art"].astype(int)
ok["releases"] = ok["releases"].astype(int)
print(f"Usable albums: {len(ok)} of {len(df)}\n")

# 3. Distribution of front-art counts
print(ok["front_art"].describe(), "\n")
print("Median:", ok["front_art"].median(), "\n")

# 4. How many albums pass at each cutoff
for c in [15, 20, 25, 30, 40]:
    print(f"front_art >= {c}: {(ok['front_art'] >= c).sum()} albums")
print("Albums with more than 100 releases (count is capped):", (ok["releases"] > 100).sum(), "\n")

# 5. Era and artist spread among albums that pass (change 25 to try other cutoffs)
passed = ok[ok["front_art"] >= 25]
print("By decade:\n", passed.groupby((passed["year"] // 10) * 10).size(), "\n")
print("Top artists:\n", passed["artist"].value_counts().head(10), "\n")

# 6. Top 30 albums by front_art
print(ok.sort_values("front_art", ascending=False)
        .head(30)[["artist", "album", "year", "front_art", "releases"]]
        .to_string(index=False))

# 7. Rows worth fixing by hand: matched, but title_ok was False
review = df[(df["status"] == "ok") & ~df["title_ok"]]
print("\nTitle mismatches to check:\n",
      review[["artist", "album", "matched_title", "front_art"]].to_string(index=False))

status
ok          438
no match     29
Name: count, dtype: int64 

Usable albums: 430 of 467

count    430.000000
mean      22.876744
std       19.197202
min        0.000000
25%        9.000000
50%       17.000000
75%       29.750000
max      100.000000
Name: front_art, dtype: float64 

Median: 17.0 

front_art >= 15: 251 albums
front_art >= 20: 188 albums
front_art >= 25: 136 albums
front_art >= 30: 108 albums
front_art >= 40: 68 albums
Albums with more than 100 releases (count is capped): 9 

By decade:
 year
1950     1
1960    28
1970    51
1980    23
1990    17
2000     5
2010     9
2020     2
dtype: int64 

Top artists:
 artist
The Beatles                    6
The Rolling Stones             6
Led Zeppelin                   5
Bruce Springsteen              5
Pink Floyd                     4
Bob Dylan                      4
Taylor Swift                   3
Black Sabbath                  3
Michael Jackson                3
The Jimi Hendrix Experience    3
Name: count, dtype: int64 

 

In [ ]:
for c in [50, 60, 70, 80, 90]:
    print(c, (ok["front_art"] >= c).sum())

In [2]:
from pathlib import Path
from collections import Counter
from PIL import Image

DATA_DIR = Path("data/assignment2/dataset")
FIX = False        # set True to re-save non-RGB / non-JPEG images as RGB JPEGs
MIN_SIDE = 150     # flag images smaller than this on either side

counts, problems, skipped = Counter(), [], []
for path in sorted(DATA_DIR.glob("*/*")):
    if path.name.startswith("."):
        continue                                   # .DS_Store etc.
    if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
        skipped.append(path)                       # manifest.csv etc.
        continue
    try:
        with Image.open(path) as im:
            im.verify()                            # catches corrupt or truncated files
        with Image.open(path) as im:               # verify() closes the file, so reopen
            fmt, mode, (w, h) = im.format, im.mode, im.size
        if min(w, h) < MIN_SIDE:
            problems.append((path, f"small: {w}x{h}"))
        if mode != "RGB" or fmt != "JPEG":
            problems.append((path, f"{fmt} {mode}" + (" -> fixed" if FIX else "")))
            if FIX:
                with Image.open(path) as im:
                    rgb = im.convert("RGB")
                rgb.save(path, "JPEG", quality=95)
        counts[path.parent.name] += 1
    except Exception as e:
        problems.append((path, f"unreadable: {e}"))

print("Images per class:")
for cls, n in sorted(counts.items()):
    print(f"  {cls:<45}{n}")
print(f"Total: {sum(counts.values())}   (skipped {len(skipped)} non-image files)\n")

print(f"{len(problems)} issues:")
for p, why in problems:
    print(f"  {p.parent.name}/{p.name}: {why}")

Images per class:
  black_sabbath__paranoid                      40
  bruce_springsteen__darkness_on_the_edge_of_town39
  carole_king__tapestry                        38
  kendrick_lamar__good_kid_m_a_a_d_city        39
  michael_jackson__thriller                    40
  miles_davis__kind_of_blue                    39
  talking_heads__remain_in_light               40
  taylor_swift__red                            39
  the_who__tommy                               40
  u2__the_joshua_tree                          39
Total: 393   (skipped 0 non-image files)

4 issues:
  michael_jackson__thriller/877ff2e7-e270-4267-a1fe-956c217f0061.jpg: PNG RGB
  michael_jackson__thriller/f7dbf356-f83a-4643-942c-8c1682d31fa6.jpg: JPEG L
  miles_davis__kind_of_blue/969e1c1c-d72b-3f05-a412-03679632cc75.jpg: JPEG CMYK
  the_who__tommy/eaba5a80-a36f-427a-b68b-580fdfde8b53.jpg: PNG RGB


In [4]:
import csv, hashlib, re, time
from pathlib import Path
import pandas as pd, requests

HEADERS = {"User-Agent": "AlbumCoverProject/0.1 (akinbobola71@gmail.com)"}
DATA, COVERS, EXTRA = Path("data/assignment2/dataset"), Path("data/raw/candidate_fronts"), Path("data/raw/remaining_fronts")

def slug(t):
    return re.sub(r"[^a-z0-9]+", "_", t.lower()).strip("_")

def mb_releases(params, retries=5):
    for attempt in range(retries):
        time.sleep(1.1 + attempt * 5)          # wait longer after each failure
        r = requests.get("https://musicbrainz.org/ws/2/release", headers=HEADERS,
                         timeout=30, params=params)
        if r.ok:
            return r.json()
        print(f"   MusicBrainz returned {r.status_code}, retrying...")
    r.raise_for_status()

screen = pd.read_csv("data/metadata/screen.csv")
screen["folder"] = screen["artist"].map(slug) + "__" + screen["album"].map(slug)
rg_of = dict(zip(screen["folder"], screen["rg_id"]))

for folder in sorted(d.name for d in DATA.iterdir() if d.is_dir()):
    dirs = [DATA / folder, COVERS / folder, EXTRA / folder]
    files = [p for d in dirs if d.exists() for p in d.glob("*.jpg")]
    seen = {p.stem for p in files}                                # releases you already have
    hashes = {hashlib.sha256(p.read_bytes()).hexdigest() for p in files}
    man = COVERS / folder / "manifest.csv"
    if man.exists():                                              # releases already checked,
        with open(man, newline="") as f:                          # incl. ones you deleted
            seen |= {r["release_id"] for r in csv.DictReader(f)}

    releases, offset = [], 0                                      # every release in the album
    while True:
        data = mb_releases({"release-group": rg_of[folder], "limit": 100,
                            "offset": offset, "fmt": "json"})

        releases += data["releases"]
        offset += 100
        if offset >= data["release-count"]:
            break

    out = EXTRA / folder
    out.mkdir(parents=True, exist_ok=True)
    new = 0
    for rel in releases:
        caa = rel.get("cover-art-archive") or {}
        if rel["id"] in seen or not caa.get("front"):
            continue
        try:
            time.sleep(1.1)
            r = requests.get(f"https://coverartarchive.org/release/{rel['id']}",
                             headers=HEADERS, timeout=30)
            r.raise_for_status()
            front = next((i for i in r.json()["images"] if i.get("front")), None)
            if not front:
                continue
            img = requests.get(front["thumbnails"].get("500") or front["image"],
                               headers=HEADERS, timeout=60)
            img.raise_for_status()
            h = hashlib.sha256(img.content).hexdigest()
            if h not in hashes:                                   # skip byte-identical copies
                (out / f"{rel['id']}.jpg").write_bytes(img.content)
                hashes.add(h)
                new += 1
        except Exception as e:
            print(f"   skipped {rel['id']}: {e}")
    have = len(list((DATA / folder).glob("*.jpg")))
    print(f"{folder}: {have} existing + {new} new = {have + new}")

black_sabbath__paranoid: 40 existing + 12 new = 52
   skipped bbbb702d-1dcd-4ef6-b8fc-734e47464c6a: 500 Server Error: Internal Server Error for url: https://dn720709.ca.archive.org/0/items/mbid-bbbb702d-1dcd-4ef6-b8fc-734e47464c6a/mbid-bbbb702d-1dcd-4ef6-b8fc-734e47464c6a-28336969374_thumb500.jpg
   skipped d4646c1f-a868-4ad8-ad5d-5904dce61f2c: 500 Server Error: Internal Server Error for url: https://dn721809.ca.archive.org/0/items/mbid-d4646c1f-a868-4ad8-ad5d-5904dce61f2c/mbid-d4646c1f-a868-4ad8-ad5d-5904dce61f2c-32580782756_thumb500.jpg
   skipped fe1541a5-11ae-4c8a-912d-c1ad5444d89e: 500 Server Error: Internal Server Error for url: https://dn710802.ca.archive.org/0/items/mbid-fe1541a5-11ae-4c8a-912d-c1ad5444d89e/mbid-fe1541a5-11ae-4c8a-912d-c1ad5444d89e-23886642278_thumb500.jpg
   skipped fb194384-f821-42e4-b6e9-e21b07d6c4a1: 500 Server Error: Internal Server Error for url: https://dn721903.ca.archive.org/0/items/mbid-fb194384-f821-42e4-b6e9-e21b07d6c4a1/index.json
bruce_springsteen

In [5]:
from pathlib import Path
import pandas as pd

SPLIT_DIR = Path("data/assignment2/split")

counts = {
    split: {d.name: len(list(d.glob("*.jpg"))) for d in sorted((SPLIT_DIR / split).iterdir()) if d.is_dir()}
    for split in ["train", "val", "test"]
}
table = pd.DataFrame(counts).fillna(0).astype(int)
table["total"] = table.sum(axis=1)
table.loc["TOTAL"] = table.sum()
print(table)

                                                 train  val  test  total
black_sabbath__paranoid                              6   28     6     40
bruce_springsteen__darkness_on_the_edge_of_town     27    6     6     39
carole_king__tapestry                               26    6     6     38
kendrick_lamar__good_kid_m_a_a_d_city               23    6    10     39
michael_jackson__thriller                           28    6     6     40
miles_davis__kind_of_blue                           27    6     6     39
talking_heads__remain_in_light                      28    6     6     40
taylor_swift__red                                   27    6     6     39
the_who__tommy                                      18   15     7     40
u2__the_joshua_tree                                 10    6    23     39
TOTAL                                              220   91    82    393
